In [ ]:
# cell 1 - init
import os, random
import numpy as np
import pandas as pd
import lightgbm as lgb
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# true for run-time, gives 180 days on one product family
DEV_MODE = False

# config
RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
pd.set_option("display.max_columns", 200)
pd.options.mode.copy_on_write = True

# style
sns.set_style("darkgrid")
plt.rc("figure", figsize=(14, 9))
plt.rc("font", size=12)

# dtypes
train_dtypes = {
    "store_nbr": "int16",
    "family": "category",
    "onpromotion": "float32",
    "sales": "float32",
}
stores_dtypes = {
    "store_nbr": "int16",
    "city": "category",
    "state": "category",
    "type": "category",
    "cluster": "int16",
}
holidays_dtypes = {
    "locale": "category",
    "locale_name": "category",
    "type": "category",
    "transferred": "bool",
}
txn_dtypes = {
    "store_nbr": "int16",
    "transactions": "int32",
}

DATA_DIR = "/kaggle/input/store-sales-time-series-forecasting"
WORK_DIR = "/kaggle/working"
BASELINE_ART_DIR = os.path.join(WORK_DIR, "artifacts_baseline_v1")
BASELINE_ART_DIR_FOLD = os.path.join(BASELINE_ART_DIR, "artifacts_baseline_perfold_v1")

ART_DIR = os.path.join(WORK_DIR, "artifacts_v1")

# artifacts
os.makedirs(BASELINE_ART_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

# load
df_train = pd.read_csv(
    "/kaggle/input/store-sales-time-series-forecasting/train.csv",
    parse_dates=["date"],
    dtype=train_dtypes,
)
oil = pd.read_csv(
    "/kaggle/input/store-sales-time-series-forecasting/oil.csv",
    parse_dates=["date"],
)
stores = pd.read_csv(
    "/kaggle/input/store-sales-time-series-forecasting/stores.csv",
    dtype=stores_dtypes,
)
holidays = pd.read_csv(
    "/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv",
    parse_dates=["date"],
    dtype=holidays_dtypes,
)
df_txn = pd.read_csv(
    "/kaggle/input/store-sales-time-series-forecasting/transactions.csv",
    parse_dates=["date"],
    dtype=txn_dtypes,
)

# basic hygiene
df_train = df_train.sort_values(["date", "store_nbr", "family"]).reset_index(drop=True)
df_train['date'] = pd.to_datetime(df_train['date'])
oil = oil.sort_values("date").reset_index(drop=True)
df_txn = df_txn.sort_values(["date", "store_nbr"]).reset_index(drop=True)
holidays = holidays.sort_values("date").reset_index(drop=True)
stores = stores.sort_values("store_nbr").reset_index(drop=True)

In [ ]:
#cell 2 - dev slice init and seasonality eda via mstl
DEV_MAX_DAYS = 365
DEV_SEED = 42

from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.stattools import acf
from scipy import signal

rng = np.random.default_rng(DEV_SEED)

train = df_train.copy()
train["date"] = train["date"].dt.to_period("D")
train = train.set_index(["date", "store_nbr", "family"]).sort_index()

counts = (
    train["sales"]
    .groupby(["store_nbr", "family"])
    .size()
    .rename("n")
    .reset_index()
)
cand = (
    train["sales"]
    .groupby(["store_nbr", "family"])
    .mean()
    .rename("mean_sales")
    .reset_index()
    .merge(counts, on=["store_nbr", "family"])
    .query("n >= 400")
    .sort_values(["mean_sales", "n"], ascending=False)
)
if cand.empty:
    cand = (
        train["sales"]
        .groupby(["store_nbr", "family"])
        .size()
        .rename("n")
        .reset_index()
        .sort_values("n", ascending=False)
    )

DEV_STORE = int(cand.iloc[0]["store_nbr"])
DEV_FAMILY = str(cand.iloc[0]["family"])

s = (
    train.xs((DEV_STORE, DEV_FAMILY), level=("store_nbr", "family"))["sales"]
    .sort_index()
)
s.index = s.index.to_timestamp()
s = s.asfreq(pd.infer_freq(s.index) or "D")
s = s.dropna()

s = s.iloc[-DEV_MAX_DAYS:]

periods = [7]
if len(s) >= 400:
    periods.append(365)

mstl = MSTL(s, periods=tuple(periods))
res = mstl.fit()
fig = res.plot()
plt.suptitle(f"mstl decomposition: store {DEV_STORE} - {DEV_FAMILY}", y=1.02)
plt.tight_layout()

s_detr = (s - pd.Series(res.trend, index=s.index)).dropna()
fs = 1.0
freqs, power = signal.welch(s_detr.values, fs=fs, nperseg=min(512, len(s_detr)))
peak_idx = np.argsort(power)[-8:][::-1]
cand_periods = (1.0 / freqs[peak_idx])
cand_periods = cand_periods[np.isfinite(cand_periods)]
print("dev slice:", {"store": DEV_STORE, "family": DEV_FAMILY, "len": int(len(s))})
print("top candidate periods (days):", np.round(cand_periods, 2))

acf_vals = acf(s_detr, nlags=60, fft=True)
lags = np.arange(len(acf_vals))
top_lags = lags[1:][np.argsort(acf_vals[1:])[-5:][::-1]]
print("top acf lags (days):", top_lags.tolist())

dev_meta = {
    "store": DEV_STORE,
    "family": DEV_FAMILY,
    "len": int(len(s)),
    "periods_used": periods,
    "top_periods_days": np.round(cand_periods, 2).tolist(),
    "top_acf_lags_days": top_lags.tolist(),
}

pd.Series(dev_meta).to_json("/kaggle/working/dev_slice_store_sales.json")

In [ ]:
#cell 3 - metrics and plots definition

# metrics
def rmsle(y_true, y_pred, clip=True, eps=1e-9):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    if clip:
        yt = np.clip(yt, 0, None)
        yp = np.clip(yp, 0, None)
    return np.sqrt(np.mean((np.log1p(yp + eps) - np.log1p(yt + eps))**2))

def smape(y_true, y_pred, eps=1e-9):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    den = (np.abs(yt) + np.abs(yp)) + eps
    return 100.0 * np.mean(np.abs(yp - yt) / den)

def wape(y_true, y_pred, eps=1e-9):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    return np.sum(np.abs(yp - yt)) / (np.sum(np.abs(yt)) + eps)

def mase_from_insample(y_true, y_pred, insample, m=7, eps=1e-9):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    ins = np.asarray(insample, dtype=float)
    if len(ins) <= m:
        mae_naive = np.mean(np.abs(yt[m:] - yt[:-m])) if len(yt) > m else np.mean(np.abs(yt - np.median(yt)))
    else:
        mae_naive = np.mean(np.abs(ins[m:] - ins[:-m]))
    return np.mean(np.abs(yp - yt)) / (mae_naive + eps)

def cost_asymm(y_true, y_pred, c_under=2.0, c_over=1.0):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    under = np.maximum(yt - yp, 0.0)
    over = np.maximum(yp - yt, 0.0)
    return c_under * np.sum(under) + c_over * np.sum(over)

def metric_bundle(y_true, y_pred, insample=None, m=7):
    out = {}
    out["rmsle"] = float(rmsle(y_true, y_pred))
    out["wape"] = float(wape(y_true, y_pred))
    out["smape"] = float(smape(y_true, y_pred))
    if insample is not None:
        out["mase"] = float(mase_from_insample(y_true, y_pred, insample, m=m))
    return out

# plots
def plot_ts(y_true_s, y_pred_s, title="forecast vs actual"):
    fig, ax = plt.subplots(figsize=(12,4))
    y_true_s.plot(ax=ax, label="actual")
    y_pred_s.plot(ax=ax, label="forecast")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    return ax

def plot_scatter(y_true, y_pred, title="pred vs actual"):
    fig, ax = plt.subplots(figsize=(5,5))
    ax.scatter(y_true, y_pred, s=10, alpha=0.5)
    lim = [0, max(np.max(y_true), np.max(y_pred)) * 1.05]
    ax.plot(lim, lim, "k--", linewidth=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("actual"); ax.set_ylabel("pred")
    ax.set_title(title)
    plt.tight_layout()
    return ax

def plot_residuals(y_true, y_pred, title="residuals"):
    res = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    fig, ax = plt.subplots(figsize=(12,3))
    ax.plot(res, linewidth=1)
    ax.axhline(0.0, color="k", linestyle="--", linewidth=1)
    ax.set_title(title)
    plt.tight_layout()
    return ax

def plot_cost_tradeoff(y_true, y_pred, ratios=None, c_over=1.0, title="cost tradeoff"):
    if ratios is None:
        ratios = np.linspace(0.5, 5.0, 30)
    costs = [cost_asymm(y_true, y_pred, c_under=r*c_over, c_over=c_over) for r in ratios]
    fig, ax = plt.subplots(figsize=(6,3))
    ax.plot(ratios, costs, marker="o", linewidth=1)
    ax.set_xlabel("under:over cost ratio")
    ax.set_ylabel("total cost")
    ax.set_title(title)
    plt.tight_layout()
    return ax


In [ ]:
#cell 4 - rolling features, safe joins, label creation (dev-aware)

df = df_train.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store_nbr","family","date"]).reset_index(drop=True)

if DEV_MODE:
    df = df[(df["store_nbr"] == DEV_STORE) & (df["family"] == DEV_FAMILY)]

cal = pd.DataFrame({"date": pd.date_range(df["date"].min(), df["date"].max(), freq="D")})

oil_feat = oil.sort_values("date").copy()
oil_feat["dcoilwm_7_ma"] = oil_feat["dcoilwtico"].rolling(7, min_periods=1).mean()
oil_feat["dcoilwm_28_ma"] = oil_feat["dcoilwtico"].rolling(28, min_periods=1).mean()
oil_feat = oil_feat.rename(columns={"dcoilwtico":"oil_price"})
oil_feat = cal.merge(oil_feat, on="date", how="left").ffill()

hol = holidays.copy()
hol["is_holiday"] = 1
hol.loc[hol["type"].isin(["Work Day"]), "is_holiday"] = 0
hol.loc[hol["transferred"] == True, "is_holiday"] = 0
hol = hol[["date","is_holiday"]].groupby("date", as_index=False).max()
hol = cal.merge(hol, on="date", how="left").fillna({"is_holiday":0})

txn = df_txn.copy().sort_values(["store_nbr","date"])
if DEV_MODE:
    txn = txn[txn["store_nbr"] == DEV_STORE]
txn["transactions"] = txn["transactions"].astype("float32")
txn["txn_lag1"] = txn.groupby("store_nbr")["transactions"].shift(1)
txn["txn_roll7_mean"] = txn.groupby("store_nbr")["transactions"].shift(1).rolling(7).mean()
txn["txn_roll28_mean"] = txn.groupby("store_nbr")["transactions"].shift(1).rolling(28).mean()
txn = txn[["store_nbr","date","txn_lag1","txn_roll7_mean","txn_roll28_mean"]]

df = df.merge(oil_feat, on="date", how="left")
df = df.merge(hol, on="date", how="left")
df = df.merge(txn, on=["store_nbr","date"], how="left")

df["dow"] = df["date"].dt.dayofweek.astype("int16")
df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int16")
df["is_month_end"] = df["date"].dt.is_month_end.astype("int8")

g = df.groupby(["store_nbr","family"], sort=False)["sales"]
for L in [7,14,28]:
    df[f"lag_{L}"] = g.shift(L)
for W in [7,28]:
    df[f"roll_{W}_mean"] = g.shift(1).rolling(W).mean()
    df[f"roll_{W}_std"]  = g.shift(1).rolling(W).std()

df["onpromotion"] = df["onpromotion"].fillna(0).astype("int16")

feat_cols = [
    "dow","weekofyear","is_month_end",
    "lag_7","lag_14","lag_28",
    "roll_7_mean","roll_7_std","roll_28_mean","roll_28_std",
    "onpromotion",
    "oil_price","dcoilwm_7_ma","dcoilwm_28_ma",
    "is_holiday",
    "txn_lag1","txn_roll7_mean","txn_roll28_mean",
]

df = df.dropna(subset=["lag_7","lag_14","lag_28","roll_7_mean","roll_28_mean"]).reset_index(drop=True)

df["store_nbr"] = df["store_nbr"].astype("category")
df["family"] = df["family"].astype("category")
cat_cols = ["store_nbr","family"]
all_features = cat_cols + feat_cols

df["y"] = np.log1p(df["sales"])

meta = {
    "dev_mode": bool(DEV_MODE),
    "store": int(DEV_STORE) if DEV_MODE else None,
    "family": str(DEV_FAMILY) if DEV_MODE else None,
    "n_rows": int(len(df)),
    "n_features": len(all_features),
}
pd.Series(meta).to_json("/kaggle/working/fe_meta_store_sales.json")
pd.Series(all_features).to_json("/kaggle/working/feature_columns.json")


In [ ]:
#cell 5 - temporal folds (rolling, leakage-safe) for 14d horizon

from datetime import timedelta

HORIZON_DAYS = 14
GAP_DAYS = 28
N_FOLDS = 5
MIN_TRAIN_DAYS = 180 if DEV_MODE else 540

def make_rolling_folds(df_in, date_col="date", horizon=14, gap=28, n_folds=3, min_train_days=180):
    dfx = df_in.reset_index(drop=True)
    dates = pd.Series(pd.to_datetime(dfx[date_col]).dt.normalize().unique()).sort_values().to_list()
    if len(dates) < (min_train_days + gap + horizon + 1):
        n_folds = 1
    folds = []
    for i in range(n_folds):
        val_start = dates[-(horizon*(n_folds - i))]
        val_end = val_start + timedelta(days=horizon - 1)
        train_end = val_start - timedelta(days=gap)
        train_start_idx = 0
        # enforce min train span
        for j, d in enumerate(dates):
            if d >= (train_end - timedelta(days=min_train_days)):
                train_start_idx = j
                break
        train_mask = (dfx[date_col] <= train_end) & (dfx[date_col] >= dates[train_start_idx])
        val_mask = (dfx[date_col] >= val_start) & (dfx[date_col] <= val_end)
        tr_idx = dfx.index[train_mask].to_numpy()
        va_idx = dfx.index[val_mask].to_numpy()
        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue
        folds.append((tr_idx, va_idx, dict(
            val_start=str(val_start.date()), val_end=str(val_end.date()),
            train_end=str(train_end.date()), gap_days=gap, horizon_days=horizon
        )))
    return folds

folds = make_rolling_folds(
    df, date_col="date",
    horizon=HORIZON_DAYS, gap=GAP_DAYS,
    n_folds=N_FOLDS, min_train_days=MIN_TRAIN_DAYS
)

fold_tbl = []
fold_id = np.full(len(df), -1, dtype=int)
for k, (tr, va, meta) in enumerate(folds, start=1):
    fold_id[va] = k
    fold_tbl.append({"fold": k, **meta, "n_train": int(len(tr)), "n_valid": int(len(va))})

df["fold_id"] = fold_id

FOLDS_DIR = os.path.join(WORK_DIR, "folds_v1")
os.makedirs(FOLDS_DIR, exist_ok=True)


# artifacts
os.makedirs(BASELINE_ART_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

pd.DataFrame(fold_tbl).to_json("/kaggle/working/folds_store_sales.json", orient="records")

In [ ]:
#cell 5a - data hygiene for modeling (fit + helpers)

import json
from pathlib import Path

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)

RARE_THRESH = 5 if DEV_MODE else 10
LOW_VAR_EPS = 1e-9

assert len(folds) > 0, "no folds found"
last_meta = folds[-1][2]
train_end_date = pd.to_datetime(last_meta["train_end"])

base_cols = list(all_features)
fit_mask = pd.to_datetime(df["date"]) <= train_end_date
df_fit = df.loc[fit_mask, base_cols].copy()

def hygiene_fit_on_train(df_train_X, rare_thresh=5):
    drop_cols, num_fill, cat_levels = [], {}, {}
    X = df_train_X.replace([np.inf, -np.inf], np.nan).copy()
    for c in X.columns:
        if str(X[c].dtype).startswith("category"):
            vc = X[c].astype("object").value_counts(dropna=False)
            keep = [k for k, v in vc.items() if v >= rare_thresh]
            cat_levels[c] = sorted(list(map(lambda z: "__OTHER__" if z is None else str(z), keep)))
        else:
            if X[c].nunique(dropna=False) <= 1:
                drop_cols.append(c); continue
            if pd.api.types.is_numeric_dtype(X[c]):
                if (X[c].std() or 0.0) < 1e-9:
                    drop_cols.append(c); continue
                med = float(np.nanmedian(X[c].values))
                num_fill[c] = med
            else:
                drop_cols.append(c)
    return {"drop_cols": sorted(drop_cols), "num_fill": num_fill, "cat_levels": cat_levels}

def hygiene_transform(X, cfg):
    Z = X.replace([np.inf, -np.inf], np.nan).copy()
    if cfg["drop_cols"]:
        Z.drop(columns=[c for c in cfg["drop_cols"] if c in Z.columns], inplace=True, errors="ignore")
    for c, med in cfg["num_fill"].items():
        if c in Z.columns:
            Z[c] = Z[c].astype(float).fillna(med)
    for c, keep_levels in cfg["cat_levels"].items():
        if c not in Z.columns: 
            continue
        Z[c] = Z[c].astype("object").apply(lambda z: "__MISSING__" if pd.isna(z) else str(z))
        Z[c] = Z[c].apply(lambda z: z if z in keep_levels else "__OTHER__")
        Z[c] = pd.Categorical(Z[c], categories=(keep_levels + ["__OTHER__","__MISSING__"]))
    return Z

HG_FIT = hygiene_fit_on_train(df_fit, rare_thresh=RARE_THRESH)
clean_preview = hygiene_transform(df_fit.head(100), HG_FIT)
CLEAN_FEATURES = list(clean_preview.columns)

with open("/kaggle/working/hygiene_config_store_sales.json","w") as f:
    json.dump(HG_FIT, f, indent=2)
pd.Series(CLEAN_FEATURES).to_json("/kaggle/working/clean_features_store_sales.json")

def get_clean_split(tr_idx, va_idx):
    X_tr = hygiene_transform(df.loc[tr_idx, base_cols], HG_FIT)
    X_va = hygiene_transform(df.loc[va_idx, base_cols], HG_FIT)
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)
    return X_tr, X_va

In [ ]:
#cell 6 - baseline lgbm vs 7d seasonal naive (eval: rmsle, wape, mase) with plots

import pickle
import lightgbm as lgb

params = dict(
    objective="regression",
    learning_rate=0.08,
    n_estimators=160,
    max_depth=8,
    num_leaves=31,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=0.5,
    min_data_in_leaf=32,
    max_bin=127,
    min_data_in_bin=5,
    feature_pre_filter=False,
    enable_bundle=False,
    deterministic=True,
    random_state=RNG_SEED,
    n_jobs=-1,
    verbosity=-1
)


assert "folds" in globals() and len(folds) > 0, "no folds found"
use_folds = folds[-min(len(folds), N_FOLDS):]

results, last_fold_panel = [], None

for k, (tr_idx, va_idx, meta) in enumerate(use_folds, start=1):
    X_tr_raw = df.loc[tr_idx, all_features]
    X_va_raw = df.loc[va_idx, all_features]

    cfg = hygiene_fit_on_train(X_tr_raw, rare_thresh=5)
    X_tr = hygiene_transform(X_tr_raw, cfg)
    X_va = hygiene_transform(X_va_raw, cfg)
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)

    y_tr_log = df.loc[tr_idx, "y"].astype(float).values
    y_va_log = df.loc[va_idx, "y"].astype(float).values
    y_va = df.loc[va_idx, "sales"].astype(float).values
    y_in = df.loc[tr_idx, "sales"].astype(float).values

    cat_features = [c for c in X_tr.columns if str(X_tr[c].dtype).startswith("category")]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr_log,
        eval_set=[(X_va, y_va_log)],
        eval_metric="rmse",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    y_pred = np.expm1(model.predict(X_va, num_iteration=model.best_iteration_))
    y_pred = np.clip(y_pred, 0, None)
    y_naive = df.loc[va_idx, "lag_7"].astype(float).clip(lower=0).values

    m_lgb = metric_bundle(y_va, y_pred, insample=y_in, m=7)
    m_nv7 = metric_bundle(y_va, y_naive, insample=y_in, m=7)

    results.append({
        "fold": k,
        "val_start": meta["val_start"],
        "val_end": meta["val_end"],
        "lgb_rmsle": m_lgb.get("rmsle"),
        "nv7_rmsle": m_nv7.get("rmsle"),
        "lgb_wape": m_lgb.get("wape"),
        "nv7_wape": m_nv7.get("wape"),
        "lgb_smape": m_lgb.get("smape"),
        "nv7_smape": m_nv7.get("smape"),
        "lgb_mase": m_lgb.get("mase"),
        "nv7_mase": m_nv7.get("mase"),
        "best_iter": int(model.best_iteration_)
    })

    fold_tag = f"fold{k}_{meta['val_start']}_to_{meta['val_end']}"
    preds = pd.DataFrame({
        "date": pd.to_datetime(df.loc[va_idx, "date"].values),
        "store_nbr": df.loc[va_idx, "store_nbr"].values,
        "family": df.loc[va_idx, "family"].values,
        "y_true": y_va,
        "y_pred_lgb": y_pred,
        "y_pred_naive7": y_naive
    }).sort_values("date")
    preds.to_csv(f"/kaggle/working/baseline_preds_{fold_tag}.csv", index=False)

    with open(f"/kaggle/working/baseline_model_{fold_tag}.pkl","wb") as f:
        pickle.dump(model, f)
    with open(f"/kaggle/working/baseline_config_{fold_tag}.json","w") as f:
        json.dump({"params":params,"features":list(X_tr.columns),"categoricals":cat_features,"meta":meta,"hygiene":cfg}, f, indent=2)

    last_fold_panel = preds.copy()

res_df = pd.DataFrame(results)
print(res_df[["fold","val_start","val_end","lgb_rmsle","nv7_rmsle","lgb_wape","nv7_wape","lgb_mase","nv7_mase","best_iter"]])

if last_fold_panel is not None:
    s_true = last_fold_panel.set_index("date")["y_true"].sort_index()
    s_lgb = last_fold_panel.set_index("date")["y_pred_lgb"].sort_index()
    s_nv7 = last_fold_panel.set_index("date")["y_pred_naive7"].sort_index()
    _ = plot_ts(s_true, s_lgb, title="lgb forecast vs actual (last fold)")
    plt.show()
    _ = plot_ts(s_true, s_nv7, title="naive-7 vs actual (last fold)")
    plt.show()
    _ = plot_scatter(s_true.values, s_lgb.values, title="pred vs actual (lgb, last fold)")
    plt.show()
    _ = plot_residuals(s_true.values, s_lgb.values, title="residuals (lgb, last fold)")
    plt.show()
    _ = plot_cost_tradeoff(s_true.values, s_lgb.values, title="cost tradeoff (lgb, last fold)")
    plt.show()

res_df.to_csv("/kaggle/working/fold_metrics_store_sales_baseline.csv", index=False)


cfg = {
    "model_type": "lgbm_regressor",
    "params": params,
    "features": list(X_tr.columns),
    "categoricals": [c for c in X_tr if X_tr[c].dtype.name == "category"],
    "dev_mode": bool(DEV_MODE)
}
with open("/kaggle/working/artifacts_baseline_v1/baseline_model_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

with open("/kaggle/working/artifacts_baseline_v1/baseline_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("artifacts:", {
    "model_pkl": "/kaggle/working/artifacts_baseline_v1/model_baseline.pkl",
    "config_json": "/kaggle/working/artifacts_baseline_v1/model_config_baseline.json"
})

In [ ]:
#cell 7 - per-store/per-family error ranking utility (mainly for DEV_MODE=False)

import glob

def _safe_smape(y, p, eps=1e-9):
    den = np.abs(y) + np.abs(p) + eps
    return 100.0 * np.mean(np.abs(p - y) / den)

def _safe_rmsle(y, p, eps=1e-9):
    y = np.clip(y, 0, None)
    p = np.clip(p, 0, None)
    return float(np.sqrt(np.mean((np.log1p(p + eps) - np.log1p(y + eps))**2)))

def build_error_ranking(pred_glob="/kaggle/working/baseline_preds_*.csv", sort_by="wape", ascending=False, min_obs=7):
    files = sorted(glob.glob(pred_glob))
    if not files:
        return pd.DataFrame()
    dfp = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in files], ignore_index=True)
    dfp["abs_err_lgb"] = (dfp["y_pred_lgb"] - dfp["y_true"]).abs()
    grp = dfp.groupby(["store_nbr","family"], as_index=False)
    out = grp.agg(
        n=("y_true","size"),
        y_sum=("y_true", lambda s: float(np.abs(s).sum())),
        ae_sum=("abs_err_lgb", "sum")
    )
    out["wape"] = np.where(out["y_sum"] > 0, out["ae_sum"] / out["y_sum"], np.nan)

    def _smape_grp(g):
        return _safe_smape(g["y_true"].values, g["y_pred_lgb"].values)
    def _rmsle_grp(g):
        return _safe_rmsle(g["y_true"].values, g["y_pred_lgb"].values)

    smape_vals = grp.apply(_smape_grp).reset_index(names="smape")
    rmsle_vals = grp.apply(_rmsle_grp).reset_index(names="rmsle")

    out = out.merge(smape_vals, on=["store_nbr","family"]).merge(rmsle_vals, on=["store_nbr","family"])
    out = out[out["n"] >= int(min_obs)].copy()
    valid_sort = sort_by if sort_by in {"wape","smape","rmsle","n"} else "wape"
    out = out.sort_values(valid_sort, ascending=ascending).reset_index(drop=True)

    cols = ["store_nbr","family","n","wape","smape","rmsle"]
    out[cols].to_csv("/kaggle/working/error_ranking_store_family.csv", index=False)
    return out[cols]

def top_k_worst(k=10, sort_by="wape", min_obs=7):
    rank = build_error_ranking(sort_by=sort_by, ascending=False, min_obs=min_obs)
    return rank.head(k)

rank_tbl = top_k_worst(k=10, sort_by="wape", min_obs=7)
print(rank_tbl.head(10))


In [ ]:
#cell 8 - ablations and feature importance

# params (same stabilized set)
ABL_PARAMS = dict(
    objective="regression",
    learning_rate=0.08,
    n_estimators=160,
    max_depth=8,
    num_leaves=31,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=0.5,
    min_data_in_leaf=32,
    max_bin=127,
    min_data_in_bin=5,
    feature_pre_filter=False,
    enable_bundle=False,
    deterministic=True,
    random_state=RNG_SEED,
    n_jobs=-1,
    verbosity=-1
)

# feature groups
CAL_FEATS = ["dow","weekofyear","is_month_end"]
OIL_FEATS = ["oil_price","dcoilwm_7_ma","dcoilwm_28_ma"]
HOL_FEATS = ["is_holiday"]
TXN_FEATS = ["txn_lag1","txn_roll7_mean","txn_roll28_mean"]
LAG_FEATS = ["lag_7","lag_14","lag_28"]
ROLL_FEATS = ["roll_7_mean","roll_7_std","roll_28_mean","roll_28_std"]
PROMO_FEATS = ["onpromotion"]

GROUPS = {
    "drop_calendar": CAL_FEATS,
    "drop_oil": OIL_FEATS,
    "drop_holiday": HOL_FEATS,
    "drop_txn": TXN_FEATS,
    "drop_roll": ROLL_FEATS,
    "drop_lag": LAG_FEATS,
    "drop_promo": PROMO_FEATS,
}

def train_eval_subset(tr_idx, va_idx, feat_subset, params):
    X_tr_raw = df.loc[tr_idx, feat_subset]
    X_va_raw = df.loc[va_idx, feat_subset]
    cfg = hygiene_fit_on_train(X_tr_raw, rare_thresh=5)
    X_tr = hygiene_transform(X_tr_raw, cfg)
    X_va = hygiene_transform(X_va_raw, cfg)
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)
    y_tr_log = df.loc[tr_idx, "y"].astype(float).values
    y_va_log = df.loc[va_idx, "y"].astype(float).values
    y_va = df.loc[va_idx, "sales"].astype(float).values
    y_in = df.loc[tr_idx, "sales"].astype(float).values
    cat_features = [c for c in X_tr.columns if str(X_tr[c].dtype).startswith("category")]
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr_log,
        eval_set=[(X_va, y_va_log)],
        eval_metric="rmse",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    y_pred = np.expm1(model.predict(X_va, num_iteration=model.best_iteration_))
    y_pred = np.clip(y_pred, 0, None)
    m = metric_bundle(y_va, y_pred, insample=y_in, m=7)
    return model, m, cfg, list(X_tr.columns)

# last 2 for runtime ease
use_folds = folds[-min(len(folds), 2):]

abl_rows = []
full_models_last_fold = {}

for k, (tr_idx, va_idx, meta) in enumerate(use_folds, start=1):
    # full model baseline
    model_full, m_full, cfg_full, used_feats_full = train_eval_subset(tr_idx, va_idx, all_features, ABL_PARAMS)
    abl_rows.append({
        "fold": k, "group": "full", "val_start": meta["val_start"], "val_end": meta["val_end"],
        "rmsle": m_full["rmsle"], "wape": m_full["wape"], "smape": m_full["smape"], "mase": m_full.get("mase", np.nan),
        "n_feats": len(used_feats_full)
    })
    if k == len(use_folds):
        full_models_last_fold = {
            "model": model_full,
            "cfg": cfg_full,
            "feats": used_feats_full,
            "meta": meta,
            "tr_idx": tr_idx,
            "va_idx": va_idx
        }

    # ablations
    for gname, gcols in GROUPS.items():
        feat_subset = [c for c in all_features if c not in gcols]
        model_sub, m_sub, cfg_sub, used_feats_sub = train_eval_subset(tr_idx, va_idx, feat_subset, ABL_PARAMS)
        abl_rows.append({
            "fold": k, "group": gname, "val_start": meta["val_start"], "val_end": meta["val_end"],
            "rmsle": m_sub["rmsle"], "wape": m_sub["wape"], "smape": m_sub["smape"], "mase": m_sub.get("mase", np.nan),
            "n_feats": len(used_feats_sub)
        })

abl_df = pd.DataFrame(abl_rows)
# deltas vs full per fold
def add_deltas(g):
    base = g.loc[g["group"]=="full", ["rmsle","wape","smape","mase"]].iloc[0]
    for m in ["rmsle","wape","smape","mase"]:
        g[f"delta_{m}"] = g[m] - base[m]
    return g

abl_df = abl_df.groupby("fold", group_keys=False).apply(add_deltas).reset_index(drop=True)
cols = ["fold","group","val_start","val_end","n_feats","rmsle","delta_rmsle","wape","delta_wape","smape","delta_smape","mase","delta_mase"]
abl_df[cols].to_csv("/kaggle/working/ablations_store_sales.csv", index=False)
print(abl_df[cols].sort_values(["fold","group"]).head(30))

# feature importance on last fold full model
imp_df = pd.DataFrame({
    "feature": full_models_last_fold["feats"],
    "gain": full_models_last_fold["model"].booster_.feature_importance(importance_type="gain"),
    "split": full_models_last_fold["model"].booster_.feature_importance(importance_type="split")
})
imp_df = imp_df.sort_values("gain", ascending=False).reset_index(drop=True)
imp_df.to_csv("/kaggle/working/feature_importance_last_fold.csv", index=False)
print(imp_df.head(20))

# simple plot top-20 by gain
top = imp_df.head(20)
plt.figure(figsize=(8,6))
plt.barh(top["feature"][::-1], top["gain"][::-1])
plt.title("feature importance (gain) - last fold")
plt.xlabel("gain")
plt.tight_layout()
plt.show()


In [ ]:
#cell 9 - aggregated cost-curve sweep across folds

def total_asymm_cost(y, p, r, c_over=1.0):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    under = np.maximum(y - p, 0.0)
    over = np.maximum(p - y, 0.0)
    return float(r * c_over * under.sum() + c_over * over.sum())

files = sorted(glob.glob("/kaggle/working/baseline_preds_*.csv"))
assert files, "no fold prediction files found"

rows = []
ratios = np.linspace(0.5, 5.0, 30)
for f in files:
    dfp = pd.read_csv(f, parse_dates=["date"])
    y = dfp["y_true"].values
    p_lgb = dfp["y_pred_lgb"].values
    p_nv7 = dfp["y_pred_naive7"].values
    for r in ratios:
        c_lgb = total_asymm_cost(y, p_lgb, r)
        c_nv7 = total_asymm_cost(y, p_nv7, r)
        rows.append({"file": f, "ratio": float(r), "cost_lgb": c_lgb, "cost_naive7": c_nv7})

cost_df = pd.DataFrame(rows)
agg = cost_df.groupby("ratio", as_index=False).agg(
    cost_lgb=("cost_lgb","sum"),
    cost_naive7=("cost_naive7","sum")
)
agg["winner"] = np.where(agg["cost_lgb"] <= agg["cost_naive7"], "lgbm", "naive7")
best_row = agg.loc[agg["cost_lgb"].idxmin()]
best_ratio = float(best_row["ratio"])
best_cost = float(best_row["cost_lgb"])

agg.to_csv("/kaggle/working/cost_sweep_agg.csv", index=False)
print("best lgbm ratio:", {"ratio": best_ratio, "total_cost": best_cost})

plt.figure(figsize=(7,4))
plt.plot(agg["ratio"], agg["cost_lgb"], label="lgbm")
plt.plot(agg["ratio"], agg["cost_naive7"], label="naive-7")
plt.xlabel("under:over cost ratio")
plt.ylabel("total cost (aggregated)")
plt.title("cost curve sweep across folds")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#cell 10 - update feature set: drop weak groups, add fourier(7) + trend deltas


eps = 1e-9

df = df.sort_values(["store_nbr","family","date"]).reset_index(drop=True)

g = df.groupby(["store_nbr","family"], sort=False)["sales"]
for W in [7,28]:
    df[f"roll_{W}_mean_tmp"] = g.shift(1).rolling(W).mean()

df["delta_roll"] = df["roll_7_mean_tmp"] - df["roll_28_mean_tmp"]
df["delta_lag"] = df["lag_7"] - df["lag_14"]
df["pct_change_7"] = (df["lag_7"] - df["lag_14"]) / (df["lag_14"] + eps)

def _ema_shifted(x, span):
    return x.shift(1).ewm(span=span, adjust=False).mean()

df["ema_7"]  = g.apply(lambda s: _ema_shifted(s, span=7)).reset_index(level=[0,1], drop=True)
df["ema_14"] = g.apply(lambda s: _ema_shifted(s, span=14)).reset_index(level=[0,1], drop=True)

df["t_ix"] = (df["date"] - df["date"].min()).dt.days.astype(float)
for k in (1,2):
    df[f"fourier7_s{k}"] = np.sin(2.0 * np.pi * k * df["t_ix"] / 7.0)
    df[f"fourier7_c{k}"] = np.cos(2.0 * np.pi * k * df["t_ix"] / 7.0)

drop_cols = [
    "weekofyear","is_month_end",
    "is_holiday",
    "oil_price","dcoilwm_7_ma","dcoilwm_28_ma",
    "roll_7_mean","roll_7_std","roll_28_mean","roll_28_std",
    "txn_roll7_mean","txn_roll28_mean"
]
for c in drop_cols:
    if c in df.columns:
        df.drop(columns=c, inplace=True)

base_keep = [
    "store_nbr","family","onpromotion",
    "lag_7","lag_14","lag_28",
    "dow",
    "txn_lag1",
    "delta_roll","delta_lag","pct_change_7",
    "ema_7","ema_14",
    "fourier7_s1","fourier7_c1","fourier7_s2","fourier7_c2"
]

df = df.dropna(subset=["lag_7","lag_14","lag_28"]).reset_index(drop=True)

df["store_nbr"] = df["store_nbr"].astype("category")
df["family"] = df["family"].astype("category")
df["onpromotion"] = df["onpromotion"].fillna(0).astype("int16")

all_features = base_keep
pd.Series(all_features).to_json("/kaggle/working/feature_columns_updated.json")


In [ ]:
#cell 11 - 5-fold training/eval with updated features + averaged importances

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)

params = dict(
    objective="regression",
    learning_rate=0.08,
    n_estimators=180,
    max_depth=8,
    num_leaves=31,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=0.5,
    min_data_in_leaf=32,
    max_bin=127,
    min_data_in_bin=5,
    feature_pre_filter=False,
    enable_bundle=False,
    deterministic=True,
    random_state=RNG_SEED,
    n_jobs=-1,
    verbosity=-1
)

results = []
imp_rows = []
last_fold_panel = None

for k, (tr_idx, va_idx, meta) in enumerate(use_folds, start=1):
    X_tr_raw = df.loc[tr_idx, all_features]
    X_va_raw = df.loc[va_idx, all_features]

    cfg = hygiene_fit_on_train(X_tr_raw, rare_thresh=5)
    X_tr = hygiene_transform(X_tr_raw, cfg)
    X_va = hygiene_transform(X_va_raw, cfg)
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)

    y_tr_log = df.loc[tr_idx, "y"].astype(float).values
    y_va_log = df.loc[va_idx, "y"].astype(float).values
    y_va = df.loc[va_idx, "sales"].astype(float).values
    y_in = df.loc[tr_idx, "sales"].astype(float).values

    cat_features = [c for c in X_tr.columns if str(X_tr[c].dtype).startswith("category")]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr_log,
        eval_set=[(X_va, y_va_log)],
        eval_metric="rmse",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    y_pred = np.expm1(model.predict(X_va, num_iteration=model.best_iteration_))
    y_pred = np.clip(y_pred, 0, None)
    y_naive = df.loc[va_idx, "lag_7"].astype(float).clip(lower=0).values

    m_lgb = metric_bundle(y_va, y_pred, insample=y_in, m=7)
    m_nv7 = metric_bundle(y_va, y_naive, insample=y_in, m=7)

    results.append({
        "fold": k,
        "val_start": meta["val_start"],
        "val_end": meta["val_end"],
        "lgb_rmsle": m_lgb.get("rmsle"),
        "nv7_rmsle": m_nv7.get("rmsle"),
        "lgb_wape": m_lgb.get("wape"),
        "nv7_wape": m_nv7.get("wape"),
        "lgb_smape": m_lgb.get("smape"),
        "nv7_smape": m_nv7.get("smape"),
        "lgb_mase": m_lgb.get("mase"),
        "nv7_mase": m_nv7.get("mase"),
        "best_iter": int(model.best_iteration_)
    })

    fold_tag = f"fold{k}_{meta['val_start']}_to_{meta['val_end']}"
    preds = pd.DataFrame({
        "date": pd.to_datetime(df.loc[va_idx, "date"].values),
        "store_nbr": df.loc[va_idx, "store_nbr"].values,
        "family": df.loc[va_idx, "family"].values,
        "y_true": y_va,
        "y_pred_lgb": y_pred,
        "y_pred_naive7": y_naive
    }).sort_values("date")
    preds.to_csv(f"/kaggle/working/preds_{fold_tag}.csv", index=False)

    with open(f"/kaggle/working/model_{fold_tag}.pkl","wb") as f:
        pickle.dump(model, f)
    with open(f"/kaggle/working/config_{fold_tag}.json","w") as f:
        json.dump({"params":params,"features":list(X_tr.columns),"categoricals":cat_features,"meta":meta,"hygiene":cfg}, f, indent=2)

    gain = model.booster_.feature_importance(importance_type="gain")
    split = model.booster_.feature_importance(importance_type="split")
    imp_rows += [{"fold": k, "feature": f, "gain": float(g), "split": int(s)} for f, g, s in zip(X_tr.columns, gain, split)]

    last_fold_panel = preds.copy()

res_df = pd.DataFrame(results)
res_df.to_csv("/kaggle/working/fold_metrics_updated_features.csv", index=False)
print(res_df[["fold","val_start","val_end","lgb_rmsle","nv7_rmsle","lgb_wape","nv7_wape","lgb_mase","nv7_mase","best_iter"]])

imp_df = pd.DataFrame(imp_rows)
imp_avg = (imp_df.groupby("feature", as_index=False)
           .agg(gain=("gain","mean"), split=("split","mean"))
           .sort_values("gain", ascending=False)
           .reset_index(drop=True))
imp_df.to_csv("/kaggle/working/feature_importance_per_fold.csv", index=False)
imp_avg.to_csv("/kaggle/working/feature_importance_avg.csv", index=False)
print(imp_avg.head(20))

if last_fold_panel is not None:
    s_true = last_fold_panel.set_index("date")["y_true"].sort_index()
    s_lgb = last_fold_panel.set_index("date")["y_pred_lgb"].sort_index()
    s_nv7 = last_fold_panel.set_index("date")["y_pred_naive7"].sort_index()
    _ = plot_ts(s_true, s_lgb, title="lgb forecast vs actual (last fold, updated features)")
    plt.show()
    _ = plot_ts(s_true, s_nv7, title="naive-7 vs actual (last fold)")
    plt.show()
    _ = plot_scatter(s_true.values, s_lgb.values, title="pred vs actual (lgb, last fold)")
    plt.show()
    _ = plot_residuals(s_true.values, s_lgb.values, title="residuals (lgb, last fold)")
    plt.show()

In [ ]:
#cell 12 - auto feature pruning (zero-gain + safe group ablation)


assert "all_features" in globals(), "missing all_features"
assert "folds" in globals() and len(folds) > 0, "no folds found"

# groups
CAL_FEATS = ["dow","weekofyear","is_month_end"]
OIL_FEATS = ["oil_price","dcoilwm_7_ma","dcoilwm_28_ma"]
HOL_FEATS = ["is_holiday"]
TXN_FEATS = ["txn_lag1","txn_roll7_mean","txn_roll28_mean"]
LAG_FEATS = ["lag_7","lag_14","lag_28"]
ROLL_FEATS = ["roll_7_mean","roll_7_std","roll_28_mean","roll_28_std","delta_roll"]
PROMO_FEATS = ["onpromotion"]
SEAS_FEATS = ["fourier7_s1","fourier7_c1","fourier7_s2","fourier7_c2"]
TREND_FEATS = ["delta_lag","pct_change_7","ema_7","ema_14"]

GROUPS = {
    "calendar": CAL_FEATS,
    "holiday": HOL_FEATS,
    "oil": OIL_FEATS,
    "txn_roll": ["txn_roll7_mean","txn_roll28_mean"],
    "txn_lag": ["txn_lag1"],
    "lags": LAG_FEATS,
    "roll": ["roll_7_mean","roll_7_std","roll_28_mean","roll_28_std"],
    "promo": PROMO_FEATS,
    "seasonal_fourier7": SEAS_FEATS,
    "trend_deltas": TREND_FEATS,
}

CORE_PROTECT = set(["family","store_nbr","lag_7","lag_14","lag_28","onpromotion", "dow"])

params = dict(
    objective="regression",
    learning_rate=0.08,
    n_estimators=160,
    max_depth=8,
    num_leaves=31,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=0.5,
    min_data_in_leaf=32,
    max_bin=127,
    min_data_in_bin=5,
    feature_pre_filter=False,
    enable_bundle=False,
    deterministic=True,
    random_state=RNG_SEED,
    n_jobs=-1,
    verbosity=-1
)

def train_eval(tr_idx, va_idx, feat_cols):
    X_tr_raw = df.loc[tr_idx, feat_cols]
    X_va_raw = df.loc[va_idx, feat_cols]
    cfg = hygiene_fit_on_train(X_tr_raw, rare_thresh=5)
    X_tr = hygiene_transform(X_tr_raw, cfg)
    X_va = hygiene_transform(X_va_raw, cfg)
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)
    y_tr = df.loc[tr_idx, "y"].astype(float).values
    y_va = df.loc[va_idx, "y"].astype(float).values
    y_true = df.loc[va_idx, "sales"].astype(float).values
    y_in = df.loc[tr_idx, "sales"].astype(float).values
    cat_features = [c for c in X_tr.columns if str(X_tr[c].dtype).startswith("category")]
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="rmse",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
    )
    pred = np.expm1(model.predict(X_va, num_iteration=model.best_iteration_))
    pred = np.clip(pred, 0, None)
    m = metric_bundle(y_true, pred, insample=y_in, m=7)
    gain = dict(zip(X_tr.columns, model.booster_.feature_importance(importance_type="gain")))
    return m, gain

use_folds = folds[-min(len(folds), 2):]

# baseline with full feature set
base_metrics = []
gain_rows = []
for (tr_idx, va_idx, meta) in use_folds:
    m, g = train_eval(tr_idx, va_idx, all_features)
    base_metrics.append(m)
    gain_rows.append(g)

base_rmsle = np.mean([x["rmsle"] for x in base_metrics])
base_wape = np.mean([x["wape"] for x in base_metrics])

# prune zero-gain across folds
avg_gain = {}
for g in gain_rows:
    for k, v in g.items():
        avg_gain[k] = avg_gain.get(k, 0.0) + float(v)
for k in list(avg_gain.keys()):
    avg_gain[k] /= float(len(gain_rows))

zero_gain = sorted([f for f, v in avg_gain.items() if (f in all_features) and (v <= 0.0) and (f not in CORE_PROTECT)])

# group ablation
group_stats = []
for gname, gcols in GROUPS.items():
    cols_to_drop = [c for c in gcols if c in all_features and c not in CORE_PROTECT]
    if not cols_to_drop:
        continue
    feat_subset = [c for c in all_features if c not in cols_to_drop]
    m_list = []
    for (tr_idx, va_idx, meta) in use_folds:
        m_sub, _ = train_eval(tr_idx, va_idx, feat_subset)
        m_list.append(m_sub)
    rmsle_mean = np.mean([x["rmsle"] for x in m_list])
    wape_mean = np.mean([x["wape"] for x in m_list])
    group_stats.append({
        "group": gname,
        "d_rmsle": rmsle_mean - base_rmsle,
        "d_wape": wape_mean - base_wape,
        "drop_cols": cols_to_drop
    })

# decide removals
removable_groups = [g["group"] for g in group_stats if (g["d_rmsle"] <= 0.0 and g["d_wape"] <= 0.0)]
cols_from_groups = []
for g in group_stats:
    if g["group"] in removable_groups:
        cols_from_groups += g["drop_cols"]

assert "zero_gain" in globals() and "cols_from_groups" in globals(), "missing pruning inputs"
assert "CORE_PROTECT" in globals() and "all_features" in globals(), "missing core or features"

pruned_set = (set(zero_gain) | set(cols_from_groups)) - set(CORE_PROTECT)
pruned_cols = sorted(list(pruned_set))
kept_cols = [c for c in all_features if c not in pruned_set]

# save and update
with open("/kaggle/working/auto_prune_report.json","w") as f:
    json.dump({
        "base_rmsle": float(base_rmsle),
        "base_wape": float(base_wape),
        "avg_gain": {k: float(v) for k,v in sorted(avg_gain.items(), key=lambda kv: -kv[1])},
        "zero_gain_features": zero_gain,
        "group_stats": group_stats,
        "removable_groups": removable_groups,
        "pruned_features": pruned_cols,
        "kept_features": kept_cols
    }, f, indent=2)

all_features = kept_cols
pd.Series(all_features).to_json("/kaggle/working/feature_columns_pruned.json")

print("base rmsle:", round(base_rmsle,6), "base wape:", round(base_wape,6))
print("zero-gain pruned:", zero_gain)
print("removable groups:", removable_groups)
print("final feature count:", len(all_features))

In [ ]:
#cell 13 - 5-fold training/eval with pruned features + refreshed importances


Path("/kaggle/working").mkdir(parents=True, exist_ok=True)

params = dict(
    objective="regression",
    learning_rate=0.08,
    n_estimators=180,
    max_depth=8,
    num_leaves=31,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=0.5,
    min_data_in_leaf=32,
    max_bin=127,
    min_data_in_bin=5,
    feature_pre_filter=False,
    enable_bundle=False,
    deterministic=True,
    random_state=RNG_SEED,
    n_jobs=-1,
    verbosity=-1
)

results = []
imp_rows = []
last_fold_panel = None

for k, (tr_idx, va_idx, meta) in enumerate(use_folds, start=1):
    X_tr_raw = df.loc[tr_idx, all_features]
    X_va_raw = df.loc[va_idx, all_features]

    cfg = hygiene_fit_on_train(X_tr_raw, rare_thresh=5)
    X_tr = hygiene_transform(X_tr_raw, cfg)
    X_va = hygiene_transform(X_va_raw, cfg)
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)

    y_tr_log = df.loc[tr_idx, "y"].astype(float).values
    y_va_log = df.loc[va_idx, "y"].astype(float).values
    y_va = df.loc[va_idx, "sales"].astype(float).values
    y_in = df.loc[tr_idx, "sales"].astype(float).values

    cat_features = [c for c in X_tr.columns if str(X_tr[c].dtype).startswith("category")]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr_log,
        eval_set=[(X_va, y_va_log)],
        eval_metric="rmse",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    y_pred = np.expm1(model.predict(X_va, num_iteration=model.best_iteration_))
    y_pred = np.clip(y_pred, 0, None)
    y_naive = df.loc[va_idx, "lag_7"].astype(float).clip(lower=0).values

    m_lgb = metric_bundle(y_va, y_pred, insample=y_in, m=7)
    m_nv7 = metric_bundle(y_va, y_naive, insample=y_in, m=7)

    results.append({
        "fold": k,
        "val_start": meta["val_start"],
        "val_end": meta["val_end"],
        "lgb_rmsle": m_lgb.get("rmsle"),
        "nv7_rmsle": m_nv7.get("rmsle"),
        "lgb_wape": m_lgb.get("wape"),
        "nv7_wape": m_nv7.get("wape"),
        "lgb_smape": m_lgb.get("smape"),
        "nv7_smape": m_nv7.get("smape"),
        "lgb_mase": m_lgb.get("mase"),
        "nv7_mase": m_nv7.get("mase"),
        "best_iter": int(model.best_iteration_)
    })

    fold_tag = f"fold{k}_{meta['val_start']}_to_{meta['val_end']}"
    preds = pd.DataFrame({
        "date": pd.to_datetime(df.loc[va_idx, "date"].values),
        "store_nbr": df.loc[va_idx, "store_nbr"].values,
        "family": df.loc[va_idx, "family"].values,
        "y_true": y_va,
        "y_pred_lgb": y_pred,
        "y_pred_naive7": y_naive
    }).sort_values("date")
    preds.to_csv(f"/kaggle/working/preds_pruned_{fold_tag}.csv", index=False)

    with open(f"/kaggle/working/model_pruned_{fold_tag}.pkl","wb") as f:
        pickle.dump(model, f)
    with open(f"/kaggle/working/config_pruned_{fold_tag}.json","w") as f:
        json.dump({"params":params,"features":list(X_tr.columns),"categoricals":cat_features,"meta":meta,"hygiene":cfg}, f, indent=2)

    gain = model.booster_.feature_importance(importance_type="gain")
    split = model.booster_.feature_importance(importance_type="split")
    imp_rows += [{"fold": k, "feature": f, "gain": float(g), "split": int(s)} for f, g, s in zip(X_tr.columns, gain, split)]

    last_fold_panel = preds.copy()

res_df = pd.DataFrame(results)
res_df.to_csv("/kaggle/working/fold_metrics_pruned.csv", index=False)
print(res_df[["fold","val_start","val_end","lgb_rmsle","nv7_rmsle","lgb_wape","nv7_wape","lgb_mase","nv7_mase","best_iter"]])

imp_df = pd.DataFrame(imp_rows)
imp_avg = (imp_df.groupby("feature", as_index=False)
           .agg(gain=("gain","mean"), split=("split","mean"))
           .sort_values("gain", ascending=False)
           .reset_index(drop=True))
imp_df.to_csv("/kaggle/working/feature_importance_per_fold_pruned.csv", index=False)
imp_avg.to_csv("/kaggle/working/feature_importance_avg_pruned.csv", index=False)
print(imp_avg.head(20))

if last_fold_panel is not None:
    s_true = last_fold_panel.set_index("date")["y_true"].sort_index()
    s_lgb = last_fold_panel.set_index("date")["y_pred_lgb"].sort_index()
    s_nv7 = last_fold_panel.set_index("date")["y_pred_naive7"].sort_index()
    _ = plot_ts(s_true, s_lgb, title="lgb forecast vs actual (last fold, pruned features)")
    plt.show()
    _ = plot_ts(s_true, s_nv7, title="naive-7 vs actual (last fold)")
    plt.show()
    _ = plot_scatter(s_true.values, s_lgb.values, title="pred vs actual (lgb, last fold)")
    plt.show()
    _ = plot_residuals(s_true.values, s_lgb.values, title="residuals (lgb, last fold)")
    plt.show()


In [ ]:
#cell 14 - quick A/B: tweedie objective vs baseline (log-target)

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)

# try to load baseline metrics (log-target run) if they exist
baseline_path = "/kaggle/working/fold_metrics_pruned.csv"
baseline_df = None
if os.path.exists(baseline_path):
    baseline_df = pd.read_csv(baseline_path)

# tweedie params
tw_params = dict(
    objective="tweedie",
    tweedie_variance_power=1.2,
    learning_rate=0.08,
    n_estimators=160,
    max_depth=8,
    num_leaves=31,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=0.5,
    min_data_in_leaf=32,
    max_bin=127,
    min_data_in_bin=5,
    feature_pre_filter=False,
    enable_bundle=False,
    deterministic=True,
    random_state=RNG_SEED,
    n_jobs=-1,
    verbosity=-1
)

tw_rows = []
for k, (tr_idx, va_idx, meta) in enumerate(use_folds, start=1):
    X_tr_raw = df.loc[tr_idx, all_features]
    X_va_raw = df.loc[va_idx, all_features]

    cfg = hygiene_fit_on_train(X_tr_raw, rare_thresh=5)
    X_tr = hygiene_transform(X_tr_raw, cfg)
    X_va = hygiene_transform(X_va_raw, cfg)
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)

    # tweedie trains on raw sales
    y_tr = df.loc[tr_idx, "sales"].astype(float).values
    y_va = df.loc[va_idx, "sales"].astype(float).values
    y_in = df.loc[tr_idx, "sales"].astype(float).values

    cat_features = [c for c in X_tr.columns if str(X_tr[c].dtype).startswith("category")]

    model = lgb.LGBMRegressor(**tw_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="rmse",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    y_pred = model.predict(X_va, num_iteration=model.best_iteration_)
    y_pred = np.clip(y_pred, 0, None)

    m_tw = metric_bundle(y_va, y_pred, insample=y_in, m=7)

    fold_tag = f"fold{k}_{meta['val_start']}_to_{meta['val_end']}"
    pd.DataFrame({
        "date": pd.to_datetime(df.loc[va_idx, "date"].values),
        "store_nbr": df.loc[va_idx, "store_nbr"].values,
        "family": df.loc[va_idx, "family"].values,
        "y_true": y_va,
        "y_pred_tweedie": y_pred
    }).sort_values("date").to_csv(f"/kaggle/working/preds_tweedie_{fold_tag}.csv", index=False)

    with open(f"/kaggle/working/model_tweedie_{fold_tag}.pkl","wb") as f:
        pickle.dump(model, f)
    with open(f"/kaggle/working/config_tweedie_{fold_tag}.json","w") as f:
        json.dump({"params":tw_params,"features":list(X_tr.columns),"categoricals":cat_features,"meta":meta,"hygiene":cfg}, f, indent=2)

    tw_rows.append({
        "fold": k,
        "val_start": meta["val_start"],
        "val_end": meta["val_end"],
        "tw_rmsle": m_tw.get("rmsle"),
        "tw_wape": m_tw.get("wape"),
        "tw_smape": m_tw.get("smape"),
        "tw_mase": m_tw.get("mase"),
        "best_iter": int(model.best_iteration_)
    })

tw_df = pd.DataFrame(tw_rows)
tw_df.to_csv("/kaggle/working/fold_metrics_tweedie.csv", index=False)

if baseline_df is not None:
    merged = (baseline_df.merge(
        tw_df.rename(columns={
            "tw_rmsle":"lgb_tweedie_rmsle",
            "tw_wape":"lgb_tweedie_wape",
            "tw_smape":"lgb_tweedie_smape",
            "tw_mase":"lgb_tweedie_mase"
        }),
        on=["fold","val_start","val_end"],
        how="inner"
    ))
    cols = [
        "fold","val_start","val_end",
        "lgb_rmsle","lgb_tweedie_rmsle",
        "lgb_wape","lgb_tweedie_wape",
        "lgb_smape","lgb_tweedie_smape",
        "lgb_mase","lgb_tweedie_mase"
    ]
    print(merged[cols])
    means = merged[cols[3:]].mean()
    print("\nmean metrics:")
    print(means)
else:
    print(tw_df)

In [ ]:
# cell 13 - model card
import textwrap

WORK = "/kaggle/working"

def try_load_json(path, default=None):
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception:
        return default

def try_load_csv(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None

metrics_path   = os.path.join(WORK, "fold_metrics_pruned.csv")
imp_avg_path   = os.path.join(WORK, "feature_importance_avg_pruned.csv")
abl_path       = os.path.join(WORK, "ablations_store_sales.csv")
cost_path      = os.path.join(WORK, "cost_sweep_agg.csv")
prune_path     = os.path.join(WORK, "auto_prune_report.json")
feats_path     = os.path.join(WORK, "feature_columns_pruned.json")
tw_path        = os.path.join(WORK, "fold_metrics_tweedie.csv")

metrics = try_load_csv(metrics_path)
imp_avg = try_load_csv(imp_avg_path)
abl     = try_load_csv(abl_path)
cost    = try_load_csv(cost_path)
prune   = try_load_json(prune_path, default={})
feats   = try_load_csv(feats_path)
tw      = try_load_csv(tw_path)

def fmt_means(df, prefix="lgb_"):
    if df is None or df.empty: 
        return "—"
    cols = [f"{prefix}{k}" for k in ["rmsle","wape","smape","mase"]]
    have = [c for c in cols if c in df.columns]
    if not have: 
        return "—"
    vals = df[have].mean().to_dict()
    return ", ".join([f"{c.split('_')[-1].upper()}={vals[c]:.6f}" for c in have])

def fmt_top_importance(df, k=10):
    if df is None or df.empty:
        return "-"
    top = df.sort_values("gain", ascending=False).head(k)
    return "\n".join([f"- {r.feature}: gain={r.gain:.4f}, split={r.split:.1f}" for r in top.itertuples(index=False)])

def fmt_ablations(df, n=12):
    if df is None or df.empty:
        return "-"
    show = df.sort_values(["fold","group"]).head(n)
    lines = ["| fold | group | d_rmsle | d_wape | d_smape | d_mase |",
             "|---:|---|---:|---:|---:|---:|"]
    for r in show.itertuples(index=False):
        lines.append(f"| {int(r.fold)} | {r.group} | {getattr(r,'delta_rmsle',np.nan):.6f} | {getattr(r,'delta_wape',np.nan):.6f} | {getattr(r,'delta_smape',np.nan):.6f} | {getattr(r,'delta_mase',np.nan):.6f} |")
    return "\n".join(lines)

def fmt_cost(cost_df):
    if cost_df is None or cost_df.empty or "cost_lgb" not in cost_df.columns:
        return "-"
    ridx = cost_df["cost_lgb"].idxmin()
    return f"best lgbm total cost at ratio {float(cost_df.loc[ridx,'ratio']):.2f}: {float(cost_df.loc[ridx,'cost_lgb']):.2f}"

kept_feats = prune.get("kept_features", [])
pruned_feats = prune.get("pruned_features", [])

card = f"""
# Model Card — Store Sales (Family-level, 14-day Horizon)

## Overview
This notebook forecasts daily sales per (store, family) over a 14-day horizon using **LightGBM** with leakage-safe temporal validation. We report **RMSLE** (benchmark) and **WAPE/SMAPE/MASE** (ops-facing), compare against a **seasonal naive-7**, and include **cost curves** and **ablations**. Runtime targets **<= 5–8 minutes** with 5 folds.

## Data
- Train tables: `train.csv`, `transactions.csv`, `holidays_events.csv`, `oil.csv`, `stores.csv`
- Unit: daily records per (store, family)
- Coverage: full dataset; dev-mode can restrict days, but folds always respect chronology

## Leakage Controls
- Temporal rolling CV: train ≤ T, **gap 28d**, validate T+1..T+14d
- All features computed **as-of t-1** (lags/rolling/ema; no future peeking)

## Features
- Kept ({len(kept_feats)}): {", ".join(kept_feats) if kept_feats else "-"}
- Pruned ({len(pruned_feats)}): {", ".join(pruned_feats) if pruned_feats else "-"}
- Rationale: auto-prune removed **zero-gain** features and any **groups** whose removal did not worsen mean RMSLE/WAPE on last 2 folds. Core signals (lags, promo, DOW if present) are protected.

## Models
- Baseline: LightGBM on **log1p(sales)** (regression)
- A/B: LightGBM **Tweedie** on raw sales (`tweedie_variance_power≈1.2`), same features/folds

## Evaluation (5-fold means)
- LGBM (log-target): {fmt_means(metrics, "lgb_")}
- Seasonal Naive-7: {fmt_means(metrics, "nv7_")}
- LGBM (tweedie): {fmt_means(tw, "tw_")}

## Ablations (delta vs full)
{fmt_ablations(abl)}

## Feature Importance (avg gain, top 10)
{fmt_top_importance(imp_avg)}

## Cost / Impact
- Under-prediction cost r:1 vs over-prediction 1; swept r∈[0.5,5.0] aggregated across folds
- {fmt_cost(cost)}

## Risks & Limits
- Drift in promotions/assortment may shift seasonality; monitor **WAPE/MASE by family**
- Intermittent demand families remain hard; consider per-family submodels for top volume

## Shipping Notes
- Inference: compute t-1 features → model → produce next-14d forecasts
- Artifacts: `preds_*.csv`, `model_*.pkl`, `config_*.json`, `feature_*_pruned.json`, `cost_sweep_agg.csv`
- Repro: deterministic seeds, hygiene config, fold jsons

## How to Run
- execute notebook cells in order; artifacts are written under `WORK`
- view card: `cat {os.path.join(WORK, 'artifacts_final_v1', 'model_card.md')}`
"""

OUT = os.path.join(WORK, "artifacts_final_v1")
os.makedirs(OUT, exist_ok=True)
with open(os.path.join(OUT, "model_card.md"), "w") as f:
    f.write(textwrap.dedent(card).strip() + "\n")

print(f"[CARD] wrote model card -> {os.path.join(OUT, 'model_card.md')}")


In [ ]:
# cell 16 - inference helper + manifest
import os, json, hashlib, pandas as pd, numpy as np
from datetime import timedelta

try:
    WORK = CFG.WORK_DIR
except Exception:
    WORK = "/kaggle/working"

ART = os.path.join(WORK, "artifacts_final_v1")
os.makedirs(ART, exist_ok=True)

def _latest_config(work_dir=WORK):
    cand = [f for f in os.listdir(work_dir) if f.startswith("config_pruned_") and f.endswith(".json")]
    if not cand:
        cand = [f for f in os.listdir(work_dir) if f.startswith("config_") and f.endswith(".json")]
    if not cand:
        raise FileNotFoundError("no config_*.json found in WORK")
    cand = sorted(cand, key=lambda x: os.path.getmtime(os.path.join(work_dir, x)))
    cfg_path = os.path.join(work_dir, cand[-1])
    if "pruned_" in cand[-1]:
        mdl_path = os.path.join(work_dir, cand[-1].replace("config_pruned_", "model_pruned_").replace(".json", ".pkl"))
    else:
        mdl_path = os.path.join(work_dir, cand[-1].replace("config_", "model_").replace(".json", ".pkl"))
    return cfg_path, mdl_path

def _alpha_from_span(span):
    return 2.0 / (span + 1.0)

def _init_panel_history(store_nbr, family, cutoff_date):
    dfx = df[(df["store_nbr"] == store_nbr) & (df["family"] == family)].copy()
    dfx = dfx.sort_values("date")
    dfx = dfx[dfx["date"] <= pd.to_datetime(cutoff_date)]
    if dfx.empty:
        raise ValueError("no history found for panel")
    return dfx

def _last_txn_lag1(store_nbr, cutoff_date):
    tx = df_txn[df_txn["store_nbr"] == store_nbr]
    tx = tx[tx["date"] <= pd.to_datetime(cutoff_date)]
    if tx.empty:
        return np.nan
    return float(tx.sort_values("date")["transactions"].iloc[-1])

def forecast_panel(store_nbr, family, cutoff_date, horizon=14, model_path=None, config_path=None, default_onpromotion=0):
    if config_path is None or model_path is None:
        config_path, model_path = _latest_config(WORK)
    with open(config_path, "r") as f:
        cfg = json.load(f)
    feats_train = cfg.get("features", [])
    hygiene_cfg = cfg.get("hygiene", {})
    cats = set(cfg.get("categoricals", []))
    import pickle, lightgbm as lgb
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    hist = _init_panel_history(store_nbr, family, cutoff_date)
    base_date0 = df["date"].min()
    txn_lag1_val = _last_txn_lag1(store_nbr, cutoff_date)

    s = hist["sales"].astype(float).reset_index(drop=True)
    if len(s) < 28:
        raise ValueError("insufficient history (<28 days) for lags")
    ema7_series = s.shift(1).ewm(span=7, adjust=False).mean()
    ema14_series = s.shift(1).ewm(span=14, adjust=False).mean()
    ema7 = float(ema7_series.iloc[-1]) if not np.isnan(ema7_series.iloc[-1]) else float(s.iloc[-7:].mean())
    ema14 = float(ema14_series.iloc[-1]) if not np.isnan(ema14_series.iloc[-1]) else float(s.iloc[-14:].mean())
    a7 = _alpha_from_span(7)
    a14 = _alpha_from_span(14)

    dates = []
    preds = []
    sales_roll = s.tolist()
    for d in range(1, horizon + 1):
        cur_date = pd.to_datetime(cutoff_date) + timedelta(days=d)
        t_ix = (cur_date - base_date0).days
        dow = int(cur_date.dayofweek)

        lag_7 = float(sales_roll[-7])
        lag_14 = float(sales_roll[-14])
        lag_28 = float(sales_roll[-28])
        delta_lag = lag_7 - lag_14
        pct_change_7 = (lag_7 - lag_14) / (lag_14 + 1e-9)

        row = {
            "store_nbr": store_nbr,
            "family": family,
            "onpromotion": int(default_onpromotion),
            "lag_7": lag_7, "lag_14": lag_14, "lag_28": lag_28,
            "txn_lag1": txn_lag1_val,
            "delta_lag": delta_lag, "pct_change_7": pct_change_7,
            "ema_7": ema7, "ema_14": ema14,
            "fourier7_s1": np.sin(2.0 * np.pi * 1 * t_ix / 7.0),
            "fourier7_c1": np.cos(2.0 * np.pi * 1 * t_ix / 7.0),
            "fourier7_s2": np.sin(2.0 * np.pi * 2 * t_ix / 7.0),
            "fourier7_c2": np.cos(2.0 * np.pi * 2 * t_ix / 7.0),
            "dow": dow
        }

        X = pd.DataFrame([row])
        for c in X.columns:
            if c in cats:
                X[c] = X[c].astype("object")
        X = hygiene_transform(X, hygiene_cfg)
        X = X.reindex(columns=feats_train, fill_value=0)

        yhat = model.predict(X, num_iteration=getattr(model, "best_iteration_", None))
        yhat = float(np.clip(yhat, 0.0, None))
        preds.append(yhat)
        dates.append(cur_date)

        sales_roll.append(yhat)
        ema7 = a7 * sales_roll[-2] + (1.0 - a7) * ema7
        ema14 = a14 * sales_roll[-2] + (1.0 - a14) * ema14

    out = pd.DataFrame({
        "date": dates,
        "store_nbr": store_nbr,
        "family": family,
        "y_pred": preds
    })
    tag = f"{int(store_nbr)}_{str(family).replace('/','-')}_{pd.to_datetime(cutoff_date).date()}"
    out_path = os.path.join(ART, f"forecast_{tag}.csv")
    out.to_csv(out_path, index=False)

    feat_hash = hashlib.md5((",".join(feats_train)).encode("utf-8")).hexdigest()
    manifest = {
        "model_path": model_path,
        "config_path": config_path,
        "feature_count": len(feats_train),
        "feature_hash": feat_hash,
        "categoricals": list(cats),
        "cutoff_date": str(pd.to_datetime(cutoff_date).date()),
        "horizon_days": int(horizon),
        "panel": {"store_nbr": int(store_nbr), "family": str(family)},
        "forecast_csv": out_path
    }
    with open(os.path.join(ART, "inference_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    print("[inference] wrote:", out_path)
    print("[manifest]  wrote:", os.path.join(ART, "inference_manifest.json"))
    return out

# example
fc = forecast_panel(store_nbr=1, family="BREAD/BAKERY", cutoff_date="2017-08-15", horizon=14)
display(fc.head())
